# 02 — SSVEP frequency analysis

We inspect the **power spectral density (PSD)** using Welch’s method and highlight the two candidate flicker frequencies (defaults: **10 Hz** and **15 Hz**).

For a quick sanity check when you do not have real SSVEP data yet, the last section builds a **pure sinusoid** mixture so you can see peaks at the target frequencies.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import matplotlib.pyplot as plt
import numpy as np

from acquisition.brainflow_stream import BrainFlowStream
from features.frequency_features import welch_psd
from signal_processing.filters import bandpass_filter
from utils.config import SSVEPConfig

In [ ]:
cfg = SSVEPConfig(project_root=ROOT)
fs = BrainFlowStream().sampling_rate()
rng = np.random.default_rng(0)
n = int(4 * fs)
t = np.arange(n) / fs

# Toy SSVEP-like signal on 3 channels (not physiologically exact — for visualization only)
f0 = cfg.left_hz
signal = np.sin(2 * np.pi * f0 * t)
noise = 0.5 * rng.standard_normal((3, n))
eeg = signal + noise

low, high = cfg.bandpass_band_hz()
eeg_f = bandpass_filter(eeg, fs, low, high)

freqs, psd = welch_psd(eeg_f, fs)

plt.figure(figsize=(8, 4))
plt.plot(freqs, psd)
plt.axvline(cfg.left_hz, color="C0", linestyle="--", label=f"LEFT {cfg.left_hz} Hz")
plt.axvline(cfg.right_hz, color="C1", linestyle="--", label=f"RIGHT {cfg.right_hz} Hz")
plt.xlim(0, 40)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Mean Welch PSD")
plt.title("PSD with SSVEP target markers")
plt.legend()
plt.tight_layout()
plt.show()

## Dominant frequency (coarse)

Below `dominant_frequency` picks the argmax below 40 Hz — useful for debugging, not a full SSVEP decoder.

In [ ]:
from features.frequency_features import dominant_frequency

print("Dominant freq (Hz):", dominant_frequency(freqs, psd))